In [1]:
# Cell 1: Build CO fermionic Hamiltonian and Hermitian fermionic terms

import pandas as pd

from openfermion.chem import MolecularData
from openfermion.ops import FermionOperator
from openfermion.transforms import get_fermion_operator, normal_ordered
from openfermion.utils import hermitian_conjugated
from openfermionpyscf import run_pyscf

pd.set_option("display.max_colwidth", None)


# ------------------------------------------------------------
# Carbon monoxide / CO configuration
# ------------------------------------------------------------

CO_BOND_LENGTH = 1.128  # Angstrom, approximate equilibrium C-O bond length
BASIS = "sto-3g"
MULTIPLICITY = 1
CHARGE = 0

# Full CO/STO-3G should give 20 qubits:
# C has 5 STO-3G spatial orbitals and O has 5 STO-3G spatial orbitals.
# 10 spatial orbitals * 2 spin orbitals = 20 qubits.
USE_ACTIVE_SPACE = True

# Optional active-space example:
# Freeze the two lowest occupied core-like spatial orbitals and keep valence orbitals active.
# This usually reduces CO/STO-3G from 20 qubits to 16 qubits.
OCCUPIED_INDICES = [0, 1]
ACTIVE_INDICES = [2, 3, 4, 5, 6, 7, 8, 9]

# Set this smaller, e.g. 1e-10, if tiny numerical terms clutter the graph.
TERM_ABS_TOL = 1e-12

# For large molecules, printing the full Hamiltonian can be very large.
PRINT_FULL_HAMILTONIAN = True


def format_fermion_term(term):
    if term == ():
        return "I"

    pieces = []
    for orbital, action in term:
        if action == 1:
            pieces.append(f"a_{orbital}^dagger")
        else:
            pieces.append(f"a_{orbital}")
    return " ".join(pieces)


def sort_fermion_key(term):
    return (len(term), term)


def coeff_to_str(c, digits=8):
    c = complex(c)
    if abs(c.imag) < 1e-12:
        return f"{c.real:+.{digits}f}"
    return f"{c.real:+.{digits}f}{c.imag:+.{digits}f}j"


def operator_to_string(op, digits=8):
    pieces = []

    for term, coeff in sorted(op.terms.items(), key=lambda item: sort_fermion_key(item[0])):
        pieces.append(f"{coeff_to_str(coeff, digits)} {format_fermion_term(term)}")

    if len(pieces) == 0:
        return "0"

    return " + ".join(pieces)


def dagger_term_key(term):
    """
    Return the OpenFermion key for O^dagger, where O is one monomial.
    """
    O = FermionOperator(term, 1.0)
    O_dag = normal_ordered(hermitian_conjugated(O))
    O_dag.compress(abs_tol=TERM_ABS_TOL)

    if len(O_dag.terms) != 1:
        raise ValueError(f"Expected one dagger term, got: {O_dag}")

    return next(iter(O_dag.terms.keys()))


def make_hermitian_fermionic_terms(fermion_hamiltonian, tol=TERM_ABS_TOL):
    """
    Group raw monomials into Hermitian fermionic Hamiltonian terms.

    If O is self-adjoint, keep c O.
    If O is not self-adjoint, group c O + c* O^dagger using the
    coefficients already present in the Hamiltonian.
    """
    used = set()
    hermitian_terms = []

    for term, coeff in fermion_hamiltonian.terms.items():
        if term in used:
            continue

        dag = dagger_term_key(term)

        if dag == term:
            T = FermionOperator(term, coeff)
            used.add(term)
        else:
            dag_coeff = fermion_hamiltonian.terms.get(dag, 0.0)

            T = FermionOperator(term, coeff)
            T += FermionOperator(dag, dag_coeff)

            used.add(term)
            used.add(dag)

        T = normal_ordered(T)
        T.compress(abs_tol=tol)
        hermitian_terms.append(T)

    return hermitian_terms


def infer_n_qubits_from_fermion_operator(op):
    """
    Infer the number of spin orbitals used by the FermionOperator.

    This is important for active-space Hamiltonians, where the active
    modes may be reindexed and smaller than molecule.n_qubits.
    """
    max_orbital = -1

    for term in op.terms:
        for orbital, action in term:
            max_orbital = max(max_orbital, orbital)

    return max_orbital + 1


def build_co_geometry(co_bond_length=CO_BOND_LENGTH):
    """
    Build linear carbon monoxide geometry.

    Carbon is placed at the origin.
    Oxygen is placed on the z-axis at the C-O bond length.
    """
    geometry = [
        ("C", (0.0, 0.0, 0.0)),
        ("O", (0.0, 0.0, co_bond_length)),
    ]

    return geometry


def build_co_fermionic_hamiltonian(
    co_bond_length=CO_BOND_LENGTH,
    basis=BASIS,
    multiplicity=MULTIPLICITY,
    charge=CHARGE,
    use_active_space=USE_ACTIVE_SPACE,
    occupied_indices=OCCUPIED_INDICES,
    active_indices=ACTIVE_INDICES,
):
    """
    Build a CO fermionic Hamiltonian using OpenFermion + PySCF.

    If use_active_space=True, molecule.get_molecular_hamiltonian is called
    with occupied_indices and active_indices.
    """
    geometry = build_co_geometry(co_bond_length=co_bond_length)

    molecule = MolecularData(
        geometry=geometry,
        basis=basis,
        multiplicity=multiplicity,
        charge=charge,
        description=f"CO_{co_bond_length}",
    )

    # FCI is not required for constructing the fermionic Hamiltonian.
    # For CO, keep run_fci=False.
    molecule = run_pyscf(
        molecule,
        run_scf=True,
        run_fci=False,
    )

    if use_active_space:
        molecular_hamiltonian = molecule.get_molecular_hamiltonian(
            occupied_indices=occupied_indices,
            active_indices=active_indices,
        )
    else:
        molecular_hamiltonian = molecule.get_molecular_hamiltonian()

    fermion_hamiltonian = get_fermion_operator(molecular_hamiltonian)
    fermion_hamiltonian = normal_ordered(fermion_hamiltonian)
    fermion_hamiltonian.compress(abs_tol=TERM_ABS_TOL)

    n_qubits = infer_n_qubits_from_fermion_operator(fermion_hamiltonian)

    return molecule, fermion_hamiltonian, n_qubits


# ------------------------------------------------------------
# Build CO fermionic Hamiltonian
# ------------------------------------------------------------

molecule, Hf, n_qubits = build_co_fermionic_hamiltonian()
hermitian_terms = make_hermitian_fermionic_terms(Hf)

print("Molecule: CO / Carbon monoxide")
print("Basis:", BASIS)
print("C-O bond length [Angstrom]:", CO_BOND_LENGTH)
print("Use active space:", USE_ACTIVE_SPACE)
if USE_ACTIVE_SPACE:
    print("Frozen occupied spatial orbitals:", OCCUPIED_INDICES)
    print("Active spatial orbitals:", ACTIVE_INDICES)
print("Full molecule electrons:", molecule.n_electrons)
print("Full molecule spatial orbitals:", molecule.n_orbitals)
print("Full molecule spin orbitals / qubits:", molecule.n_qubits)
print("Hamiltonian spin orbitals / qubits used:", n_qubits)
print("Number of raw OpenFermion monomial terms:", len(Hf.terms))
print("Number of Hermitian fermionic terms:", len(hermitian_terms))

if PRINT_FULL_HAMILTONIAN:
    print("\n=== Full fermionic Hamiltonian H_f ===")
    print(Hf)


# ------------------------------------------------------------
# Tables
# ------------------------------------------------------------

raw_rows = []

for idx, (term, coeff) in enumerate(
    sorted(Hf.terms.items(), key=lambda item: sort_fermion_key(item[0]))
):
    raw_rows.append(
        {
            "raw_index": idx,
            "coefficient": coeff_to_str(coeff),
            "monomial": format_fermion_term(term),
            "OpenFermion_key": term,
        }
    )

raw_df = pd.DataFrame(raw_rows)

print("\n=== Raw fermionic monomials c_alpha O_alpha ===")
display(raw_df)


hermitian_rows = []

for i, T in enumerate(hermitian_terms):
    hermitian_rows.append(
        {
            "vertex": f"T_{i}",
            "number_of_monomials": len(T.terms),
            "fermionic_term": operator_to_string(T),
        }
    )

hermitian_df = pd.DataFrame(hermitian_rows)

print("\n=== Hermitian fermionic terms T_i ===")
display(hermitian_df)

Molecule: CO / Carbon monoxide
Basis: sto-3g
C-O bond length [Angstrom]: 1.128
Use active space: True
Frozen occupied spatial orbitals: [0, 1]
Active spatial orbitals: [2, 3, 4, 5, 6, 7, 8, 9]
Full molecule electrons: 14
Full molecule spatial orbitals: 10
Full molecule spin orbitals / qubits: 20
Hamiltonian spin orbitals / qubits used: 16
Number of raw OpenFermion monomial terms: 2329
Number of Hermitian fermionic terms: 1233

=== Full fermionic Hamiltonian H_f ===
-79.18038708155783 [] +
-6.773231995171077 [0^ 0] +
0.021650539286147463 [0^ 2] +
0.37087205324728245 [0^ 8] +
0.5405841908917532 [0^ 14] +
-0.7838835182489181 [1^ 0^ 1 0] +
-0.030456901348131095 [1^ 0^ 2 1] +
0.030456901348131095 [1^ 0^ 3 0] +
-0.09839388932423647 [1^ 0^ 3 2] +
-0.12457614244631218 [1^ 0^ 5 4] +
-0.12457614244631218 [1^ 0^ 7 6] +
-0.0861023176728764 [1^ 0^ 8 1] +
0.030631752313312055 [1^ 0^ 8 3] +
0.0861023176728764 [1^ 0^ 9 0] +
-0.030631752313312055 [1^ 0^ 9 2] +
-0.03352897581812794 [1^ 0^ 9 8] +
-0.0361

,raw_index,coefficient,monomial,OpenFermion_key
0,0,-79.18038708,I,()
1,1,-6.77323200,a_0^dagger a_0,"((0, 1), (0, 0))"
2,2,+0.02165054,a_0^dagger a_2,"((0, 1), (2, 0))"
3,3,+0.37087205,a_0^dagger a_8,"((0, 1), (8, 0))"
4,4,+0.54058419,a_0^dagger a_14,"((0, 1), (14, 0))"
...,...,...,...,...
2324,2324,+0.02893774,a_15^dagger a_14^dagger a_14 a_9,"((15, 1), (14, 1), (14, 0), (9, 0))"
2325,2325,+0.05423272,a_15^dagger a_14^dagger a_15 a_0,"((15, 1), (14, 1), (15, 0), (0, 0))"
2326,2326,+0.01937035,a_15^dagger a_14^dagger a_15 a_2,"((15, 1), (14, 1), (15, 0), (2, 0))"
2327,2327,-0.02893774,a_15^dagger a_14^dagger a_15 a_8,"((15, 1), (14, 1), (15, 0), (8, 0))"



=== Hermitian fermionic terms T_i ===


,vertex,number_of_monomials,fermionic_term
0,T_0,1,-79.18038708 I
1,T_1,1,-6.77323200 a_0^dagger a_0
2,T_2,2,+0.02165054 a_0^dagger a_2 + +0.02165054 a_2^dagger a_0
3,T_3,2,+0.37087205 a_0^dagger a_8 + +0.37087205 a_8^dagger a_0
4,T_4,2,+0.54058419 a_0^dagger a_14 + +0.54058419 a_14^dagger a_0
...,...,...,...
1228,T_1228,2,+0.04517971 a_14^dagger a_13^dagger a_15 a_12 + +0.04517971 a_15^dagger a_12^dagger a_14 a_13
1229,T_1229,1,-0.58962539 a_15^dagger a_12^dagger a_15 a_12
1230,T_1230,1,-0.58962539 a_14^dagger a_13^dagger a_14 a_13
1231,T_1231,1,-0.54444568 a_15^dagger a_13^dagger a_15 a_13


In [2]:
# Cell 2: Build the H2 fermionic noncommutation graph

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from openfermion.transforms import normal_ordered

pd.set_option("display.max_colwidth", None)


def fermionic_commutator(A, B, tol=1e-12):
    """
    Compute [A, B] = AB - BA directly in the fermionic algebra.
    """
    C = normal_ordered(A * B - B * A)
    C.compress(abs_tol=tol)
    return C


def commute(A, B, tol=1e-12):
    """
    Return True if [A, B] = 0.
    """
    C = fermionic_commutator(A, B, tol=tol)
    return len(C.terms) == 0


# ------------------------------------------------------------
# Build noncommutation graph
# ------------------------------------------------------------
# Vertex i = Hermitian fermionic term T_i
# Edge (i, j) exists if [T_i, T_j] != 0

G = nx.Graph()

for i, T in enumerate(hermitian_terms):
    G.add_node(
        i,
        label=f"T_{i}",
        operator=T,
        operator_string=operator_to_string(T),
        number_of_monomials=len(T.terms),
    )

for i in range(len(hermitian_terms)):
    for j in range(i + 1, len(hermitian_terms)):
        Cij = fermionic_commutator(hermitian_terms[i], hermitian_terms[j])

        if len(Cij.terms) != 0:
            G.add_edge(
                i,
                j,
                commutator=Cij,
                commutator_string=operator_to_string(Cij),
            )


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

n_vertices = G.number_of_nodes()
n_total_pairs = n_vertices * (n_vertices - 1) // 2
n_noncommuting_pairs = G.number_of_edges()
n_commuting_pairs = n_total_pairs - n_noncommuting_pairs

print("=== Fermionic noncommutation graph summary ===")
print("Number of vertices / fermionic terms:", n_vertices)
print("Number of total unordered pairs:", n_total_pairs)
print("Number of noncommuting pairs / edges:", n_noncommuting_pairs)
print("Number of commuting pairs:", n_commuting_pairs)
print("Is graph bipartite?", nx.is_bipartite(G))


# ------------------------------------------------------------
# Vertex table
# ------------------------------------------------------------

vertex_rows = []

for i, data in G.nodes(data=True):
    vertex_rows.append(
        {
            "vertex": f"T_{i}",
            "degree": G.degree[i],
            "commutes_with_all": G.degree[i] == 0,
            "fermionic_term": data["operator_string"],
        }
    )

vertex_df = pd.DataFrame(vertex_rows)

print("\n=== Vertices: fermionic terms ===")
display(vertex_df)


# ------------------------------------------------------------
# Edge table
# ------------------------------------------------------------

edge_rows = []

for i, j, data in G.edges(data=True):
    edge_rows.append(
        {
            "source": f"T_{i}",
            "target": f"T_{j}",
            "meaning": f"[T_{i}, T_{j}] != 0",
            "commutator": data["commutator_string"],
        }
    )

edge_df = pd.DataFrame(edge_rows)

print("\n=== Edges: noncommuting pairs ===")
display(edge_df)


# # ------------------------------------------------------------
# # Draw graph
# # ------------------------------------------------------------

# plt.figure(figsize=(12, 8))

# pos = nx.kamada_kawai_layout(G)

# node_labels = {
#     i: f"T_{i}"
#     for i in G.nodes()
# }

# node_sizes = [
#     1000 + 250 * G.degree[i]
#     for i in G.nodes()
# ]

# nx.draw_networkx_nodes(G, pos, node_size=node_sizes)
# nx.draw_networkx_edges(G, pos, width=1.5)
# nx.draw_networkx_labels(G, pos, labels=node_labels, font_size=11, font_weight="bold")

# plt.title("H2 Fermionic Noncommutation Graph")
# plt.axis("off")
# plt.show()

=== Fermionic noncommutation graph summary ===
Number of vertices / fermionic terms: 1233
Number of total unordered pairs: 759528
Number of noncommuting pairs / edges: 402448
Number of commuting pairs: 357080
Is graph bipartite? False

=== Vertices: fermionic terms ===


,vertex,degree,commutes_with_all,fermionic_term
0,T_0,0,True,-79.18038708 I
1,T_1,240,False,-6.77323200 a_0^dagger a_0
2,T_2,470,False,+0.02165054 a_0^dagger a_2 + +0.02165054 a_2^dagger a_0
3,T_3,470,False,+0.37087205 a_0^dagger a_8 + +0.37087205 a_8^dagger a_0
4,T_4,470,False,+0.54058419 a_0^dagger a_14 + +0.54058419 a_14^dagger a_0
...,...,...,...,...
1228,T_1228,732,False,+0.04517971 a_14^dagger a_13^dagger a_15 a_12 + +0.04517971 a_15^dagger a_12^dagger a_14 a_13
1229,T_1229,415,False,-0.58962539 a_15^dagger a_12^dagger a_15 a_12
1230,T_1230,415,False,-0.58962539 a_14^dagger a_13^dagger a_14 a_13
1231,T_1231,391,False,-0.54444568 a_15^dagger a_13^dagger a_15 a_13



=== Edges: noncommuting pairs ===


,source,target,meaning,commutator
0,T_1,T_2,"[T_1, T_2] != 0",-0.14664413 a_0^dagger a_2 + +0.14664413 a_2^dagger a_0
1,T_1,T_3,"[T_1, T_3] != 0",-2.51200246 a_0^dagger a_8 + +2.51200246 a_8^dagger a_0
2,T_1,T_4,"[T_1, T_4] != 0",-3.66150214 a_0^dagger a_14 + +3.66150214 a_14^dagger a_0
3,T_1,T_38,"[T_1, T_38] != 0",+0.20629166 a_1^dagger a_0^dagger a_2 a_1 + -0.20629166 a_2^dagger a_1^dagger a_1 a_0
4,T_1,T_39,"[T_1, T_39] != 0",+0.58319097 a_1^dagger a_0^dagger a_8 a_1 + -0.58319097 a_8^dagger a_1^dagger a_1 a_0
...,...,...,...,...
402443,T_1224,T_1228,"[T_1224, T_1228] != 0",-0.02459790 a_14^dagger a_13^dagger a_11^dagger a_15 a_12 a_11 + +0.02459790 a_15^dagger a_12^dagger a_11^dagger a_14 a_13 a_11
402444,T_1225,T_1226,"[T_1225, T_1226] != 0",-0.02680107 a_13^dagger a_12^dagger a_15 a_14 + +0.02680107 a_15^dagger a_14^dagger a_13 a_12
402445,T_1226,T_1232,"[T_1226, T_1232] != 0",-0.03203655 a_13^dagger a_12^dagger a_15 a_14 + +0.03203655 a_15^dagger a_14^dagger a_13 a_12
402446,T_1228,T_1229,"[T_1228, T_1229] != 0",+0.02663910 a_14^dagger a_13^dagger a_15 a_12 + -0.02663910 a_15^dagger a_12^dagger a_14 a_13


In [3]:
# Cell 2 alpha: Faster H2 / molecular fermionic noncommutation graph
#
# Main idea:
#   1. Use cheap fermionic index rules first.
#   2. Only if rules cannot decide, use exact OpenFermion symbolic commutator.
#   3. Never build sparse/dense matrices.
#
# This cell assumes Cell 1 already defined:
#   hermitian_terms
#   operator_to_string

import time
from collections import Counter

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from openfermion.utils import commutator
from openfermion.transforms import normal_ordered

pd.set_option("display.max_colwidth", None)


# ------------------------------------------------------------
# Fermionic key utilities
# ------------------------------------------------------------

def key_modes(key):
    """
    Modes appearing in one OpenFermion monomial key.

    Example:
        ((3, 1), (0, 1), (3, 0), (0, 0)) -> {0, 3}
    """
    return frozenset(mode for mode, action in key)


def key_creations(key):
    """
    Creation modes in one monomial.
    """
    return frozenset(mode for mode, action in key if action == 1)


def key_annihilations(key):
    """
    Annihilation modes in one monomial.
    """
    return frozenset(mode for mode, action in key if action == 0)


def key_net_delta(key):
    """
    Net occupation change caused by a monomial.

    creation contributes +1
    annihilation contributes -1

    For example:
        a_2^dagger a_1^dagger a_3 a_0
    has delta:
        +1 on modes 2 and 1
        -1 on modes 3 and 0
    """
    delta = Counter()

    for mode, action in key:
        if action == 1:
            delta[mode] += 1
        else:
            delta[mode] -= 1

    return delta


def is_diagonal_key(key):
    """
    True if a monomial preserves occupation mode-by-mode.

    Examples:
        a_p^dagger a_p is diagonal.
        a_p^dagger a_q^dagger a_q a_p is diagonal.
        a_p^dagger a_q is not diagonal when p != q.
    """
    delta = key_net_delta(key)
    return all(value == 0 for value in delta.values())


def is_even_key(key):
    """
    Electronic Hamiltonian terms normally have even fermionic parity:
    length 0, 2, or 4.
    """
    return len(key) % 2 == 0


# ------------------------------------------------------------
# Safe monomial-level commutation rules
# ------------------------------------------------------------

def diagonal_key_commutes_with_key(diagonal_key, other_key):
    """
    Safe rule:

    A diagonal occupation operator depending on modes S commutes with another
    monomial if the other monomial has zero net occupation change on every
    mode in S.
    """
    support = key_modes(diagonal_key)
    delta = key_net_delta(other_key)

    return all(delta.get(mode, 0) == 0 for mode in support)


def no_cross_contractions_even_commute(key_a, key_b):
    """
    Safe rule for normal-ordered even fermionic monomials.

    If there are no possible cross contractions:
        annihilations(A) intersect creations(B) = empty
        annihilations(B) intersect creations(A) = empty

    then even monomials commute.

    This catches many cases beyond completely disjoint support.
    """
    if not is_even_key(key_a) or not is_even_key(key_b):
        return False

    a_ann = key_annihilations(key_a)
    a_cre = key_creations(key_a)

    b_ann = key_annihilations(key_b)
    b_cre = key_creations(key_b)

    return a_ann.isdisjoint(b_cre) and b_ann.isdisjoint(a_cre)


def monomial_pair_definitely_commutes(key_a, key_b):
    """
    Return (True, reason) only when we are sure two monomials commute.
    Return (False, None) if the rule cannot decide.

    Important:
        False here does NOT mean noncommuting.
        It only means "unknown; use exact symbolic fallback."
    """
    # Identity commutes with everything.
    if key_a == () or key_b == ():
        return True, "identity"

    # Any monomial commutes with itself.
    if key_a == key_b:
        return True, "same_monomial"

    # Diagonal occupation-like monomials commute with each other.
    if is_diagonal_key(key_a) and is_diagonal_key(key_b):
        return True, "diagonal_diagonal"

    # Diagonal with excitation-like term, if excitation preserves diagonal support.
    if is_diagonal_key(key_a) and diagonal_key_commutes_with_key(key_a, key_b):
        return True, "diagonal_support_preserved"

    if is_diagonal_key(key_b) and diagonal_key_commutes_with_key(key_b, key_a):
        return True, "diagonal_support_preserved"

    # Even monomials with no cross contractions commute.
    if no_cross_contractions_even_commute(key_a, key_b):
        return True, "no_cross_contractions_even"

    return False, None


# ------------------------------------------------------------
# Operator-level metadata and precheck
# ------------------------------------------------------------

def operator_metadata(op):
    """
    Precompute simple structural data for one FermionOperator.
    """
    keys = list(op.terms.keys())

    modes = set()
    for key in keys:
        modes.update(key_modes(key))

    return {
        "is_zero": len(keys) == 0,
        "only_identity": len(keys) == 1 and keys[0] == (),
        "modes": frozenset(modes),
        "is_even": all(is_even_key(key) for key in keys),
        "is_diagonal": all(is_diagonal_key(key) for key in keys),
        "number_of_monomials": len(keys),
    }


def operator_pair_definitely_commutes(A, B, meta_A, meta_B):
    """
    Return (True, reason) only for guaranteed-commuting pairs.
    Return (False, None) when unresolved.
    """
    if meta_A["is_zero"] or meta_B["is_zero"]:
        return True, "zero"

    if meta_A["only_identity"] or meta_B["only_identity"]:
        return True, "identity"

    # Very cheap global rule:
    # disjoint even fermionic operators commute.
    if (
        meta_A["is_even"]
        and meta_B["is_even"]
        and meta_A["modes"].isdisjoint(meta_B["modes"])
    ):
        return True, "disjoint_even_support"

    # Diagonal occupation-like operators commute with each other.
    if meta_A["is_diagonal"] and meta_B["is_diagonal"]:
        return True, "diagonal_diagonal"

    # More detailed but still cheap:
    # if every monomial pair has a safe commuting reason, the sums commute.
    reasons = Counter()

    for key_a in A.terms:
        for key_b in B.terms:
            ok, reason = monomial_pair_definitely_commutes(key_a, key_b)

            if not ok:
                return False, None

            reasons[reason] += 1

    if len(reasons) > 0:
        main_reason = reasons.most_common(1)[0][0]
        return True, f"all_monomial_pairs_{main_reason}"

    return False, None


# ------------------------------------------------------------
# Exact symbolic fallback
# ------------------------------------------------------------

def exact_symbolic_fermionic_commutator(A, B, tol=1e-12):
    """
    Exact symbolic commutator in fermionic algebra.

    This does not build a 2^n matrix.
    """
    C = normal_ordered(commutator(A, B))
    C.compress(abs_tol=tol)
    return C


# ------------------------------------------------------------
# Alpha graph builder
# ------------------------------------------------------------

def build_fermionic_noncommutation_graph_alpha(
    hermitian_terms,
    tol=1e-12,
    store_commutators=False,
):
    """
    Build noncommutation graph using:
        fast safe index rules first,
        exact symbolic OpenFermion fallback only when needed.

    Parameters
    ----------
    hermitian_terms:
        list of FermionOperator terms T_i from Cell 1.

    tol:
        numerical compression tolerance.

    store_commutators:
        False is recommended for large molecules.
        True is useful for H2 debugging, but can be memory-heavy.

    Returns
    -------
    G:
        networkx.Graph

    stats_df:
        pandas.DataFrame with timing and skip counts
    """
    t_start = time.perf_counter()

    G = nx.Graph()
    stats = Counter()

    metadata = [operator_metadata(T) for T in hermitian_terms]

    # Add vertices.
    for i, T in enumerate(hermitian_terms):
        G.add_node(
            i,
            label=f"T_{i}",
            operator=T,
            operator_string=operator_to_string(T),
            number_of_monomials=len(T.terms),
            modes=sorted(metadata[i]["modes"]),
            is_diagonal=metadata[i]["is_diagonal"],
            is_even=metadata[i]["is_even"],
        )

    n = len(hermitian_terms)

    # Pairwise graph construction.
    for i in range(n):
        A = hermitian_terms[i]
        meta_A = metadata[i]

        for j in range(i + 1, n):
            B = hermitian_terms[j]
            meta_B = metadata[j]

            stats["total_pairs"] += 1

            # 1. Fast guaranteed-commuting rules.
            definitely_commutes, reason = operator_pair_definitely_commutes(
                A, B, meta_A, meta_B
            )

            if definitely_commutes:
                stats["pairs_skipped_by_index_rules"] += 1
                stats[f"skip_{reason}"] += 1
                continue

            # 2. Exact symbolic fallback.
            stats["pairs_sent_to_exact_symbolic"] += 1

            Cij = exact_symbolic_fermionic_commutator(A, B, tol=tol)

            if len(Cij.terms) != 0:
                stats["noncommuting_edges"] += 1

                edge_data = {
                    "method": "exact_symbolic_fallback",
                    "meaning": f"[T_{i}, T_{j}] != 0",
                }

                if store_commutators:
                    edge_data["commutator"] = Cij
                    edge_data["commutator_string"] = operator_to_string(Cij)
                else:
                    edge_data["commutator_string"] = (
                        "(not stored; rerun with store_commutators=True)"
                    )

                G.add_edge(i, j, **edge_data)

            else:
                stats["exact_symbolic_found_commuting"] += 1

    elapsed = time.perf_counter() - t_start

    stats["vertices"] = n
    stats["edges"] = G.number_of_edges()
    stats["commuting_pairs"] = stats["total_pairs"] - G.number_of_edges()
    stats["elapsed_seconds"] = elapsed

    stats_df = pd.DataFrame([dict(stats)])

    return G, stats_df


# ------------------------------------------------------------
# Run alpha graph builder
# ------------------------------------------------------------

# For H2 debugging, you can set store_commutators=True.
# For larger molecules, keep this False.
G_alpha, stats_df = build_fermionic_noncommutation_graph_alpha(
    hermitian_terms,
    tol=1e-12,
    store_commutators=False,
)

print("=== Fermionic noncommutation graph alpha summary ===")
display(stats_df)

print("Number of vertices / fermionic terms:", G_alpha.number_of_nodes())
print("Number of noncommuting pairs / edges:", G_alpha.number_of_edges())
print("Is graph bipartite?", nx.is_bipartite(G_alpha))


# ------------------------------------------------------------
# Vertex table
# ------------------------------------------------------------

vertex_rows = []

for i, data in G_alpha.nodes(data=True):
    vertex_rows.append(
        {
            "vertex": f"T_{i}",
            "degree": G_alpha.degree[i],
            "commutes_with_all": G_alpha.degree[i] == 0,
            "number_of_monomials": data["number_of_monomials"],
            "modes": data["modes"],
            "is_diagonal": data["is_diagonal"],
            "fermionic_term": data["operator_string"],
        }
    )

vertex_df_alpha = pd.DataFrame(vertex_rows)

print("\n=== Alpha vertices: fermionic terms ===")
display(vertex_df_alpha)


# ------------------------------------------------------------
# Edge table
# ------------------------------------------------------------

edge_rows = []

for i, j, data in G_alpha.edges(data=True):
    edge_rows.append(
        {
            "source": f"T_{i}",
            "target": f"T_{j}",
            "meaning": data["meaning"],
            "method": data["method"],
            "commutator": data["commutator_string"],
        }
    )

edge_df_alpha = pd.DataFrame(edge_rows)

print("\n=== Alpha edges: noncommuting pairs ===")
display(edge_df_alpha)

# The commutation graph is too large for this.

# # ------------------------------------------------------------
# # Draw graph
# # ------------------------------------------------------------

# plt.figure(figsize=(12, 8))

# pos = nx.kamada_kawai_layout(G_alpha)

# node_labels = {
#     i: f"T_{i}"
#     for i in G_alpha.nodes()
# }

# node_sizes = [
#     1000 + 250 * G_alpha.degree[i]
#     for i in G_alpha.nodes()
# ]

# nx.draw_networkx_nodes(G_alpha, pos, node_size=node_sizes)
# nx.draw_networkx_edges(G_alpha, pos, width=1.5)
# nx.draw_networkx_labels(
#     G_alpha,
#     pos,
#     labels=node_labels,
#     font_size=11,
#     font_weight="bold",
# )

# plt.title("Fermionic Noncommutation Graph Alpha")
# plt.axis("off")
# plt.show()

=== Fermionic noncommutation graph alpha summary ===


,total_pairs,pairs_skipped_by_index_rules,skip_identity,pairs_sent_to_exact_symbolic,noncommuting_edges,skip_disjoint_even_support,skip_diagonal_diagonal,skip_all_monomial_pairs_diagonal_support_preserved,exact_symbolic_found_commuting,vertices,edges,commuting_pairs,elapsed_seconds
0,759528,316528,1232,443000,402448,309232,1920,4144,40552,1233,402448,357080,24.203536


Number of vertices / fermionic terms: 1233
Number of noncommuting pairs / edges: 402448
Is graph bipartite? False

=== Alpha vertices: fermionic terms ===


,vertex,degree,commutes_with_all,number_of_monomials,modes,is_diagonal,fermionic_term
0,T_0,0,True,1,[],True,-79.18038708 I
1,T_1,240,False,1,[0],True,-6.77323200 a_0^dagger a_0
2,T_2,470,False,2,"[0, 2]",False,+0.02165054 a_0^dagger a_2 + +0.02165054 a_2^dagger a_0
3,T_3,470,False,2,"[0, 8]",False,+0.37087205 a_0^dagger a_8 + +0.37087205 a_8^dagger a_0
4,T_4,470,False,2,"[0, 14]",False,+0.54058419 a_0^dagger a_14 + +0.54058419 a_14^dagger a_0
...,...,...,...,...,...,...,...
1228,T_1228,732,False,2,"[12, 13, 14, 15]",False,+0.04517971 a_14^dagger a_13^dagger a_15 a_12 + +0.04517971 a_15^dagger a_12^dagger a_14 a_13
1229,T_1229,415,False,1,"[12, 15]",True,-0.58962539 a_15^dagger a_12^dagger a_15 a_12
1230,T_1230,415,False,1,"[13, 14]",True,-0.58962539 a_14^dagger a_13^dagger a_14 a_13
1231,T_1231,391,False,1,"[13, 15]",True,-0.54444568 a_15^dagger a_13^dagger a_15 a_13



=== Alpha edges: noncommuting pairs ===


,source,target,meaning,method,commutator
0,T_1,T_2,"[T_1, T_2] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)
1,T_1,T_3,"[T_1, T_3] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)
2,T_1,T_4,"[T_1, T_4] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)
3,T_1,T_38,"[T_1, T_38] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)
4,T_1,T_39,"[T_1, T_39] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)
...,...,...,...,...,...
402443,T_1224,T_1228,"[T_1224, T_1228] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)
402444,T_1225,T_1226,"[T_1225, T_1226] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)
402445,T_1226,T_1232,"[T_1226, T_1232] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)
402446,T_1228,T_1229,"[T_1228, T_1229] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)


In [4]:
print("Same edge set?")
print(set(G.edges()) == set(G_alpha.edges()))

Same edge set?
True


In [5]:
# Cell 3: Color graph, build commuting blocks, map JW/BK, verify commutation

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from itertools import combinations
from openfermion.ops import FermionOperator
from openfermion.transforms import normal_ordered, jordan_wigner, bravyi_kitaev

pd.set_option("display.max_colwidth", None)


# ------------------------------------------------------------
# 1. Color the noncommutation graph
# ------------------------------------------------------------
# Since edges mean noncommutation, each color class is a commuting group.

coloring = nx.coloring.greedy_color(G, strategy="largest_first")

color_groups = {}

for node, color in coloring.items():
    color_groups.setdefault(color, []).append(node)

for color in color_groups:
    color_groups[color] = sorted(color_groups[color])

color_names = [
    "red",
    "blue",
    "green",
    "orange",
    "purple",
    "brown",
    "pink",
    "gray",
]

color_name = {
    color: color_names[color] if color < len(color_names) else f"color_{color}"
    for color in color_groups
}

num_grouped_terms = sum(len(nodes) for nodes in color_groups.values())

print("Number of fermionic terms / vertices:", G.number_of_nodes())
print("Number of colors / commuting groups:", len(color_groups))
print("Number of grouped terms:", num_grouped_terms)

assert num_grouped_terms == G.number_of_nodes()


# ------------------------------------------------------------
# 2. Verify each color group is mutually commuting
# ------------------------------------------------------------

def verify_commuting_group(nodes, tol=1e-12):
    for i, j in combinations(nodes, 2):
        A = G.nodes[i]["operator"]
        B = G.nodes[j]["operator"]

        if not commute(A, B, tol=tol):
            return False

    return True


group_summary_rows = []

for color, nodes in sorted(color_groups.items()):
    group_summary_rows.append(
        {
            "color_id": color,
            "color_name": color_name[color],
            "number_of_terms": len(nodes),
            "vertices": [f"T_{i}" for i in nodes],
            "verified_mutually_commuting": verify_commuting_group(nodes),
        }
    )

group_summary_df = pd.DataFrame(group_summary_rows)

print("\n=== Commuting groups from graph coloring ===")
display(group_summary_df)


# ------------------------------------------------------------
# 3. Build Hamiltonian pieces by color
# ------------------------------------------------------------

H_by_color = {}

for color, nodes in sorted(color_groups.items()):
    H_color = FermionOperator.zero()

    for node in nodes:
        H_color += G.nodes[node]["operator"]

    H_color = normal_ordered(H_color)
    H_color.compress(abs_tol=1e-12)

    H_by_color[color] = H_color


color_block_rows = []

for color, nodes in sorted(color_groups.items()):
    for local_index, node in enumerate(nodes, start=1):
        color_block_rows.append(
            {
                "color_block": f"H_{color_name[color]}",
                "local_term_name": f"{color_name[color][0].upper()}_{local_index}",
                "vertex": f"T_{node}",
                "fermionic_term": G.nodes[node]["operator_string"],
            }
        )

color_block_df = pd.DataFrame(color_block_rows)

print("\n=== Which T_i belongs to which color block ===")
display(color_block_df)


print("\n=== Hamiltonian split by commuting color groups ===")

for color, H_color in H_by_color.items():
    nodes = color_groups[color]

    print("\n" + "=" * 80)
    print(f"H_{color_name[color]} consists of:")
    print(" + ".join([f"T_{node}" for node in nodes]))

    print(f"\nSummed operator H_{color_name[color]} =")
    print(H_color)


# ------------------------------------------------------------
# 4. Trotter ordering induced by color groups
# ------------------------------------------------------------

trotter_order = []

for color, nodes in sorted(color_groups.items()):
    for node in nodes:
        trotter_order.append(node)

print("\n=== Trotter order by commuting color groups ===")
print([f"T_{i}" for i in trotter_order])


# ------------------------------------------------------------
# 5. Helper functions for JW/BK output
# ------------------------------------------------------------

def sort_qubit_key(term):
    return (len(term), term)


def format_qubit_term(term):
    if term == ():
        return "I"

    return " ".join([f"{pauli}{qubit}" for qubit, pauli in term])


def qubit_coeff_to_str(c, digits=8):
    c = complex(c)
    if abs(c.imag) < 1e-12:
        return f"{c.real:+.{digits}f}"
    return f"{c.real:+.{digits}f}{c.imag:+.{digits}f}j"


def qubit_operator_to_string(op, digits=8):
    pieces = []

    for term, coeff in sorted(op.terms.items(), key=lambda item: sort_qubit_key(item[0])):
        pieces.append(f"{qubit_coeff_to_str(coeff, digits)} {format_qubit_term(term)}")

    if len(pieces) == 0:
        return "0"

    return " + ".join(pieces)


def apply_bk(op, n_qubits):
    try:
        return bravyi_kitaev(op, n_qubits=n_qubits)
    except TypeError:
        return bravyi_kitaev(op, n_qubits)


def qubit_commutator(A, B, tol=1e-12):
    C = A * B - B * A
    C.compress(abs_tol=tol)
    return C


def qubit_commute(A, B, tol=1e-12):
    C = qubit_commutator(A, B, tol=tol)
    return len(C.terms) == 0


n_qubits = molecule.n_qubits

print("\nNumber of qubits / spin orbitals:", n_qubits)


# ------------------------------------------------------------
# 6. Map each fermionic vertex T_i to JW(T_i) and BK(T_i)
# ------------------------------------------------------------

mapped_rows = []

for node in sorted(G.nodes()):
    T_i = G.nodes[node]["operator"]

    JW_T_i = jordan_wigner(T_i)
    JW_T_i.compress(abs_tol=1e-12)

    BK_T_i = apply_bk(T_i, n_qubits=n_qubits)
    BK_T_i.compress(abs_tol=1e-12)

    G.nodes[node]["JW_operator"] = JW_T_i
    G.nodes[node]["BK_operator"] = BK_T_i
    G.nodes[node]["JW_operator_string"] = qubit_operator_to_string(JW_T_i)
    G.nodes[node]["BK_operator_string"] = qubit_operator_to_string(BK_T_i)

    node_color = coloring[node]
    node_color_name = color_name[node_color]

    mapped_rows.append(
        {
            "color": node_color_name,
            "vertex": f"T_{node}",
            "fermionic_term": G.nodes[node]["operator_string"],
            "number_of_JW_Pauli_strings": len(JW_T_i.terms),
            "JW_transform": qubit_operator_to_string(JW_T_i),
            "number_of_BK_Pauli_strings": len(BK_T_i.terms),
            "BK_transform": qubit_operator_to_string(BK_T_i),
        }
    )

mapped_terms_df = pd.DataFrame(mapped_rows)

print("\n=== Fermionic terms mapped to JW and BK ===")
display(mapped_terms_df)


# ------------------------------------------------------------
# 7. Map each color block H_color to JW and BK
# ------------------------------------------------------------

JW_by_color = {}
BK_by_color = {}

block_rows = []

for color, H_color in sorted(H_by_color.items()):
    JW_color = jordan_wigner(H_color)
    JW_color.compress(abs_tol=1e-12)

    BK_color = apply_bk(H_color, n_qubits=n_qubits)
    BK_color.compress(abs_tol=1e-12)

    JW_by_color[color] = JW_color
    BK_by_color[color] = BK_color

    block_rows.append(
        {
            "color_block": f"H_{color_name[color]}",
            "fermionic_vertices": " + ".join([f"T_{node}" for node in color_groups[color]]),
            "number_of_fermionic_terms": len(color_groups[color]),
            "number_of_JW_Pauli_strings": len(JW_color.terms),
            "JW_block": qubit_operator_to_string(JW_color),
            "number_of_BK_Pauli_strings": len(BK_color.terms),
            "BK_block": qubit_operator_to_string(BK_color),
        }
    )

block_map_df = pd.DataFrame(block_rows)

print("\n=== Color blocks mapped to JW and BK ===")
display(block_map_df)


# ------------------------------------------------------------
# 8. Verify same-color terms commute after JW and BK
# ------------------------------------------------------------

verification_rows = []

for color, nodes in sorted(color_groups.items()):
    for i, j in combinations(nodes, 2):
        Ti = G.nodes[i]["operator"]
        Tj = G.nodes[j]["operator"]

        JW_Ti = G.nodes[i]["JW_operator"]
        JW_Tj = G.nodes[j]["JW_operator"]

        BK_Ti = G.nodes[i]["BK_operator"]
        BK_Tj = G.nodes[j]["BK_operator"]

        verification_rows.append(
            {
                "color_group": color_name[color],
                "pair": f"T_{i}, T_{j}",
                "fermionic_commute": commute(Ti, Tj),
                "JW_commute": qubit_commute(JW_Ti, JW_Tj),
                "BK_commute": qubit_commute(BK_Ti, BK_Tj),
            }
        )

verification_df = pd.DataFrame(verification_rows)

print("\n=== Verify commuting groups after JW/BK mapping ===")
display(verification_df)

Number of fermionic terms / vertices: 1233
Number of colors / commuting groups: 101
Number of grouped terms: 1233

=== Commuting groups from graph coloring ===


,color_id,color_name,number_of_terms,vertices,verified_mutually_commuting
0,0,red,13,"[T_0, T_42, T_74, T_75, T_339, T_838, T_858, T_859, T_969, T_1178, T_1192, T_1213, T_1216]",True
1,1,blue,12,"[T_45, T_110, T_114, T_369, T_532, T_582, T_583, T_746, T_1175, T_1181, T_1226, T_1228]",True
2,2,green,12,"[T_48, T_147, T_150, T_403, T_529, T_554, T_556, T_721, T_1176, T_1187, T_1214, T_1219]",True
3,3,orange,12,"[T_57, T_213, T_221, T_461, T_543, T_655, T_660, T_809, T_841, T_879, T_1052, T_1107]",True
4,4,purple,12,"[T_60, T_246, T_253, T_490, T_540, T_631, T_637, T_789, T_842, T_880, T_1051, T_1106]",True
...,...,...,...,...,...
96,96,color_96,8,"[T_176, T_275, T_875, T_1039, T_1056, T_1171, T_1195, T_1209]",True
97,97,color_97,11,"[T_120, T_266, T_559, T_668, T_941, T_965, T_1032, T_1104, T_1158, T_1188, T_1230]",True
98,98,color_98,9,"[T_234, T_645, T_855, T_966, T_1074, T_1146, T_1182, T_1220, T_1223]",True
99,99,color_99,9,"[T_156, T_586, T_856, T_919, T_967, T_1025, T_1061, T_1114, T_1117]",True



=== Which T_i belongs to which color block ===


,color_block,local_term_name,vertex,fermionic_term
0,H_red,R_1,T_0,-79.18038708 I
1,H_red,R_2,T_42,-0.09839389 a_1^dagger a_0^dagger a_3 a_2 + -0.09839389 a_3^dagger a_2^dagger a_1 a_0
2,H_red,R_3,T_74,-0.05275202 a_2^dagger a_0^dagger a_14 a_8 + -0.05275202 a_14^dagger a_8^dagger a_2 a_0
3,H_red,R_4,T_75,+0.09839389 a_2^dagger a_1^dagger a_3 a_0 + +0.09839389 a_3^dagger a_0^dagger a_2 a_1
4,H_red,R_5,T_339,-0.05275202 a_3^dagger a_1^dagger a_15 a_9 + -0.05275202 a_15^dagger a_9^dagger a_3 a_1
...,...,...,...,...
1228,H_color_99,C_9,T_1117,-0.44640929 a_8^dagger a_7^dagger a_8 a_7
1229,H_color_100,C_1,T_857,-0.01594568 a_6^dagger a_4^dagger a_12 a_6 + -0.01594568 a_12^dagger a_6^dagger a_6 a_4
1230,H_color_100,C_2,T_896,+0.00166829 a_10^dagger a_4^dagger a_12 a_10 + +0.00166829 a_12^dagger a_10^dagger a_10 a_4
1231,H_color_100,C_3,T_968,-0.01594568 a_7^dagger a_5^dagger a_13 a_7 + -0.01594568 a_13^dagger a_7^dagger a_7 a_5



=== Hamiltonian split by commuting color groups ===

H_red consists of:
T_0 + T_42 + T_74 + T_75 + T_339 + T_838 + T_858 + T_859 + T_969 + T_1178 + T_1192 + T_1213 + T_1216

Summed operator H_red =
-79.18038708155783 [] +
-0.09839388932423647 [1^ 0^ 3 2] +
-0.05275202292511176 [2^ 0^ 14 8] +
0.09839388932423647 [2^ 1^ 3 0] +
0.09839388932423647 [3^ 0^ 2 1] +
-0.05275202292511176 [3^ 1^ 15 9] +
-0.09839388932423647 [3^ 2^ 1 0] +
-0.029540279440861017 [5^ 4^ 7 6] +
-0.11509142653211227 [6^ 4^ 12 10] +
0.029540279440861017 [6^ 5^ 7 4] +
0.029540279440861017 [7^ 4^ 6 5] +
-0.11509142653211227 [7^ 5^ 13 11] +
-0.029540279440861017 [7^ 6^ 5 4] +
-0.08170342510710048 [9^ 8^ 15 14] +
-0.025921807427223022 [11^ 10^ 13 12] +
-0.11509142653211228 [12^ 10^ 6 4] +
0.025921807427223022 [12^ 11^ 13 10] +
0.025921807427223022 [13^ 10^ 12 11] +
-0.11509142653211228 [13^ 11^ 7 5] +
-0.025921807427223022 [13^ 12^ 11 10] +
-0.05275202292511176 [14^ 8^ 2 0] +
0.08170342510710048 [14^ 9^ 15 8] +
0.08170342

,color,vertex,fermionic_term,number_of_JW_Pauli_strings,JW_transform,number_of_BK_Pauli_strings,BK_transform
0,red,T_0,-79.18038708 I,1,-79.18038708 I,1,-79.18038708 I
1,color_86,T_1,-6.77323200 a_0^dagger a_0,2,-3.38661600 I + +3.38661600 Z0,2,-3.38661600 I + +3.38661600 Z0
2,color_61,T_2,+0.02165054 a_0^dagger a_2 + +0.02165054 a_2^dagger a_0,2,+0.01082527 X0 Z1 X2 + +0.01082527 Y0 Z1 Y2,2,+0.01082527 X0 Y1 Y2 + -0.01082527 Y0 Y1 X2
3,color_82,T_3,+0.37087205 a_0^dagger a_8 + +0.37087205 a_8^dagger a_0,2,+0.18543603 X0 Z1 Z2 Z3 Z4 Z5 Z6 Z7 X8 + +0.18543603 Y0 Z1 Z2 Z3 Z4 Z5 Z6 Z7 Y8,2,+0.18543603 X0 X1 X3 Y7 Y8 X9 X11 + -0.18543603 Y0 X1 X3 Y7 X8 X9 X11
4,color_83,T_4,+0.54058419 a_0^dagger a_14 + +0.54058419 a_14^dagger a_0,2,+0.27029210 X0 Z1 Z2 Z3 Z4 Z5 Z6 Z7 Z8 Z9 Z10 Z11 Z12 Z13 X14 + +0.27029210 Y0 Z1 Z2 Z3 Z4 Z5 Z6 Z7 Z8 Z9 Z10 Z11 Z12 Z13 Y14,2,+0.27029210 X0 X1 X3 Y7 Z11 Z13 Y14 + -0.27029210 Y0 X1 X3 Y7 Z11 Z13 X14
...,...,...,...,...,...,...,...
1228,blue,T_1228,+0.04517971 a_14^dagger a_13^dagger a_15 a_12 + +0.04517971 a_15^dagger a_12^dagger a_14 a_13,8,-0.00564746 X12 X13 X14 X15 + -0.00564746 X12 X13 Y14 Y15 + -0.00564746 X12 Y13 X14 Y15 + +0.00564746 X12 Y13 Y14 X15 + +0.00564746 Y12 X13 X14 Y15 + -0.00564746 Y12 X13 Y14 X15 + -0.00564746 Y12 Y13 X14 X15 + -0.00564746 Y12 Y13 Y14 Y15,8,-0.00564746 X12 X14 + -0.00564746 Y12 Y14 + +0.00564746 X12 Z13 X14 + +0.00564746 Y12 Z13 Y14 + -0.00564746 Z7 Z11 X12 X14 Z15 + -0.00564746 Z7 Z11 Y12 Y14 Z15 + +0.00564746 Z7 Z11 X12 Z13 X14 Z15 + +0.00564746 Z7 Z11 Y12 Z13 Y14 Z15
1229,color_25,T_1229,-0.58962539 a_15^dagger a_12^dagger a_15 a_12,4,+0.14740635 I + -0.14740635 Z12 + -0.14740635 Z15 + +0.14740635 Z12 Z15,4,+0.14740635 I + -0.14740635 Z12 + -0.14740635 Z7 Z11 Z13 Z14 Z15 + +0.14740635 Z7 Z11 Z12 Z13 Z14 Z15
1230,color_97,T_1230,-0.58962539 a_14^dagger a_13^dagger a_14 a_13,4,+0.14740635 I + -0.14740635 Z13 + -0.14740635 Z14 + +0.14740635 Z13 Z14,4,+0.14740635 I + -0.14740635 Z14 + -0.14740635 Z12 Z13 + +0.14740635 Z12 Z13 Z14
1231,color_17,T_1231,-0.54444568 a_15^dagger a_13^dagger a_15 a_13,4,+0.13611142 I + -0.13611142 Z13 + -0.13611142 Z15 + +0.13611142 Z13 Z15,4,+0.13611142 I + -0.13611142 Z12 Z13 + +0.13611142 Z7 Z11 Z12 Z14 Z15 + -0.13611142 Z7 Z11 Z13 Z14 Z15



=== Color blocks mapped to JW and BK ===


,color_block,fermionic_vertices,number_of_fermionic_terms,number_of_JW_Pauli_strings,JW_block,number_of_BK_Pauli_strings,BK_block
0,H_red,T_0 + T_42 + T_74 + T_75 + T_339 + T_838 + T_858 + T_859 + T_969 + T_1178 + T_1192 + T_1213 + T_1216,13,49,-79.18038708 I + -0.02459847 X0 X1 Y2 Y3 + +0.02459847 X0 Y1 Y2 X3 + +0.02459847 Y0 X1 X2 Y3 + -0.02459847 Y0 Y1 X2 X3 + -0.00738507 X4 X5 Y6 Y7 + +0.00738507 X4 Y5 Y6 X7 + +0.00738507 Y4 X5 X6 Y7 + -0.00738507 Y4 Y5 X6 X7 + -0.02042586 X8 X9 Y14 Y15 + +0.02042586 X8 Y9 Y14 X15 + +0.02042586 Y8 X9 X14 Y15 + -0.02042586 Y8 Y9 X14 X15 + -0.00648045 X10 X11 Y12 Y13 + +0.00648045 X10 Y11 Y12 X13 + +0.00648045 Y10 X11 X12 Y13 + -0.00648045 Y10 Y11 X12 X13 + +0.01438643 X4 Z5 X6 X10 Z11 X12 + -0.01438643 X4 Z5 X6 Y10 Z11 Y12 + +0.01438643 X4 Z5 Y6 X10 Z11 Y12 + +0.01438643 X4 Z5 Y6 Y10 Z11 X12 + +0.01438643 Y4 Z5 X6 X10 Z11 Y12 + +0.01438643 Y4 Z5 X6 Y10 Z11 X12 + -0.01438643 Y4 Z5 Y6 X10 Z11 X12 + +0.01438643 Y4 Z5 Y6 Y10 Z11 Y12 + +0.01438643 X5 Z6 X7 X11 Z12 X13 + -0.01438643 X5 Z6 X7 Y11 Z12 Y13 + +0.01438643 X5 Z6 Y7 X11 Z12 Y13 + +0.01438643 X5 Z6 Y7 Y11 Z12 X13 + +0.01438643 Y5 Z6 X7 X11 Z12 Y13 + +0.01438643 Y5 Z6 X7 Y11 Z12 X13 + -0.01438643 Y5 Z6 Y7 X11 Z12 X13 + +0.01438643 Y5 Z6 Y7 Y11 Z12 Y13 + +0.00659400 X0 Z1 X2 X8 Z9 Z10 Z11 Z12 Z13 X14 + -0.00659400 X0 Z1 X2 Y8 Z9 Z10 Z11 Z12 Z13 Y14 + +0.00659400 X0 Z1 Y2 X8 Z9 Z10 Z11 Z12 Z13 Y14 + +0.00659400 X0 Z1 Y2 Y8 Z9 Z10 Z11 Z12 Z13 X14 + +0.00659400 Y0 Z1 X2 X8 Z9 Z10 Z11 Z12 Z13 Y14 + +0.00659400 Y0 Z1 X2 Y8 Z9 Z10 Z11 Z12 Z13 X14 + -0.00659400 Y0 Z1 Y2 X8 Z9 Z10 Z11 Z12 Z13 X14 + +0.00659400 Y0 Z1 Y2 Y8 Z9 Z10 Z11 Z12 Z13 Y14 + +0.00659400 X1 Z2 X3 X9 Z10 Z11 Z12 Z13 Z14 X15 + -0.00659400 X1 Z2 X3 Y9 Z10 Z11 Z12 Z13 Z14 Y15 + +0.00659400 X1 Z2 Y3 X9 Z10 Z11 Z12 Z13 Z14 Y15 + +0.00659400 X1 Z2 Y3 Y9 Z10 Z11 Z12 Z13 Z14 X15 + +0.00659400 Y1 Z2 X3 X9 Z10 Z11 Z12 Z13 Z14 Y15 + +0.00659400 Y1 Z2 X3 Y9 Z10 Z11 Z12 Z13 Z14 X15 + -0.00659400 Y1 Z2 Y3 X9 Z10 Z11 Z12 Z13 Z14 X15 + +0.00659400 Y1 Z2 Y3 Y9 Z10 Z11 Z12 Z13 Z14 Y15,49,-79.18038708 I + +0.02459847 X0 Z1 X2 + +0.02459847 Y0 Z1 Y2 + +0.00738507 X4 Z5 X6 + +0.00738507 Y4 Z5 Y6 + +0.02042586 X8 Z9 X14 + +0.02042586 Y8 Z9 Y14 + +0.00648045 X10 X12 Z13 + +0.00648045 Y10 Y12 Z13 + +0.02459847 X0 Z1 X2 Z3 + +0.02459847 Y0 Z1 Y2 Z3 + +0.00648045 Z9 X10 Z11 X12 + +0.00648045 Z9 Y10 Z11 Y12 + +0.00738507 Z3 X4 Z5 X6 Z7 + +0.00738507 Z3 Y4 Z5 Y6 Z7 + +0.01438643 Z3 Y5 Z7 X11 Y13 + +0.01438643 Z4 Y5 Z6 X11 Y13 + +0.01438643 X5 Z6 X11 Z12 X13 + -0.00659400 X1 Z2 Y9 Y11 Z13 Z14 + +0.00659400 Y1 Z3 Z7 Y9 X11 Z15 + -0.01438643 X5 Z6 Z9 Z10 Y11 Y13 + +0.02042586 Z7 X8 Z11 Z13 X14 Z15 + +0.02042586 Z7 Y8 Z11 Z13 Y14 Z15 + -0.00659400 Z0 X1 Z3 Y9 Y11 Z13 Z14 + +0.00659400 Z0 Y1 Z2 Z7 Y9 X11 Z15 + +0.00659400 X1 Z2 Z7 Z8 X9 X11 Z15 + +0.00659400 Y1 Z3 Z8 X9 Y11 Z13 Z14 + +0.01438643 Z3 Z4 X5 Z7 X11 Z12 X13 + +0.00659400 X0 Y1 X2 X8 X9 Y11 Z13 X14 + -0.00659400 X0 Y1 X2 Y8 X9 Y11 Z13 Y14 + +0.00659400 X0 Y1 Y2 X8 X9 Y11 Z13 Y14 + +0.00659400 X0 Y1 Y2 Y8 X9 Y11 Z13 X14 + +0.00659400 Y0 Y1 X2 X8 X9 Y11 Z13 Y14 + +0.00659400 Y0 Y1 X2 Y8 X9 Y11 Z13 X14 + -0.00659400 Y0 Y1 Y2 X8 X9 Y11 Z13 X14 + +0.00659400 Y0 Y1 Y2 Y8 X9 Y11 Z13 Y14 + +0.00659400 Z0 X1 Z3 Z7 Z8 X9 X11 Z15 + +0.00659400 Z0 Y1 Z2 Z8 X9 Y11 Z13 Z14 + -0.01438643 Z3 Z4 X5 Z7 Z9 Z10 Y11 Y13 + +0.01438643 Z3 Y5 Z7 Z9 Z10 Y11 Z12 X13 + +0.01438643 X4 Y5 X6 Z9 X10 Y11 X12 X13 + -0.01438643 X4 Y5 X6 Z9 Y10 Y11 Y12 X13 + +0.01438643 X4 Y5 Y6 Z9 X10 Y11 Y12 X13 + +0.01438643 X4 Y5 Y6 Z9 Y10 Y11 X12 X13 + +0.01438643 Y4 Y5 X6 Z9 X10 Y11 Y12 X13 + +0.01438643 Y4 Y5 X6 Z9 Y10 Y11 X12 X13 + -0.01438643 Y4 Y5 Y6 Z9 X10 Y11 X12 X13 + +0.01438643 Y4 Y5 Y6 Z9 Y10 Y11 Y12 X13 + +0.01438643 Z4 Y5 Z6 Z9 Z10 Y11 Z12 X13
1,H_blue,T_45 + T_110 + T_114 + T_369 + T_532 + T_582 + T_583 + T_746 + T_1175 + T_1181 + T_1226 + T_1228,12,48,-0.03114404 X0 X1 Y4 Y5 + +0.03114404 X0 Y1 Y4 X5 + +0.03114404 Y0 X1 X4 Y5 + -0.03114404 Y0 Y1 X4 X5 + -0.01282173 X2 X3 Y6 Y7 + 


=== Verify commuting groups after JW/BK mapping ===


,color_group,pair,fermionic_commute,JW_commute,BK_commute
0,red,"T_0, T_42",True,True,True
1,red,"T_0, T_74",True,True,True
2,red,"T_0, T_75",True,True,True
3,red,"T_0, T_339",True,True,True
4,red,"T_0, T_838",True,True,True
...,...,...,...,...,...
8175,color_100,"T_857, T_968",True,True,True
8176,color_100,"T_857, T_1006",True,True,True
8177,color_100,"T_896, T_968",True,True,True
8178,color_100,"T_896, T_1006",True,True,True


In [6]:
# Find duplicated JW Pauli strings across fermionic terms T_i

from collections import defaultdict
import pandas as pd

pauli_usage = defaultdict(list)

for node in sorted(G.nodes()):
    JW_T = G.nodes[node]["JW_operator"]

    for pauli_key, coeff in JW_T.terms.items():
        pauli_string = format_qubit_term(pauli_key)

        pauli_usage[pauli_string].append(
            {
                "vertex": f"T_{node}",
                "coefficient": coeff,
                "fermionic_term": G.nodes[node]["operator_string"],
            }
        )

duplicate_rows = []

for pauli_string, appearances in pauli_usage.items():
    if len(appearances) > 1:
        duplicate_rows.append(
            {
                "JW_Pauli_string": pauli_string,
                "number_of_appearances": len(appearances),
                "appears_in_vertices": [x["vertex"] for x in appearances],
                "coefficients": [x["coefficient"] for x in appearances],
            }
        )

duplicate_jw_df = pd.DataFrame(duplicate_rows)
duplicate_jw_df = duplicate_jw_df.sort_values(
    "number_of_appearances",
    ascending=False
).reset_index(drop=True)

print("Total JW Pauli-string appearances:", sum(len(G.nodes[node]["JW_operator"].terms) for node in G.nodes()))
print("Number of unique JW Pauli strings:", len(pauli_usage))
print("Number of duplicated JW Pauli strings:", len(duplicate_jw_df))

display(duplicate_jw_df)

Total JW Pauli-string appearances: 7977
Number of unique JW Pauli strings: 3689
Number of duplicated JW Pauli strings: 2977


,JW_Pauli_string,number_of_appearances,appears_in_vertices,coefficients
0,I,137,"[T_0, T_1, T_5, T_9, T_12, T_15, T_18, T_21, T_24, T_27, T_29, T_31, T_32, T_33, T_34, T_35, T_36, T_37, T_65, T_78, T_102, T_120, T_138, T_156, T_174, T_195, T_209, T_234, T_241, T_266, T_273, T_302, T_306, T_330, T_343, T_361, T_376, T_394, T_415, T_429, T_450, T_457, T_479, T_486, T_514, T_518, T_526, T_547, T_559, T_574, T_586, T_601, T_616, T_628, T_645, T_651, T_668, T_674, T_696, T_699, T_714, T_723, T_738, T_753, T_765, T_779, T_785, T_799, T_805, T_826, T_829, T_835, T_853, T_862, T_873, T_881, T_891, T_905, T_915, T_931, T_937, T_950, T_953, T_964, T_970, T_980, T_991, T_1001, T_1015, T_1021, T_1032, T_1035, T_1040, T_1053, T_1061, T_1071, T_1079, T_1087, T_1096, T_1101, ...]","[-79.18038708155783, -3.3866159975855386, -3.3866159975855386, -2.8471053474780827, -2.8471053474780827, -2.844898468653311, -2.844898468653311, -2.8448984686533123, -2.8448984686533123, -2.2442892707457056, -2.2442892707457056, -2.3583722598550354, -2.3583722598550354, -2.3583722598550363, -2.3583722598550363, -2.367041902369186, -2.367041902369186, 0.19597087956222953, 0.13293527926023815, 0.15753375159129723, 0.139211099222233, 0.17035513483381104, 0.13921109922223307, 0.1703551348338111, 0.1071600158191113, 0.1155422597736433, 0.13025744866044014, 0.13995040339462847, 0.1302574486604402, 0.13995040339462855, 0.14805676163167628, 0.17359014345771293, 0.15753375159129723, 0.13293527926023815, 0.17035513483381104, 0.139211099222233, 0.1703551348338111, 0.13921109922223307, 0.1155422597736433, 0.1071600158191113, 0.13995040339462847, 0.13025744866044014, 0.13995040339462855, 0.1302574486604402, 0.17359014345771293, 0.14805676163167628, 0.17811696595829796, 0.13999632264745243, 0.15281805234930942, 0.13999632264745246, 0.15281805234930945, 0.0835254057322433, 0.110941292194991, 0.11421199645458707, 0.12832548556368464, 0.11421199645458717, 0.12832548556368473, 0.1294816515181696, 0.14407388478161626, 0.15281805234930942, 0.13999632264745243, 0.15281805234930945, 0.13999632264745246, 0.110941292194991, 0.0835254057322433, 0.12832548556368464, 0.11421199645458707, 0.12832548556368473, 0.11421199645458717, 0.14407388478161626, 0.1294816515181696, 0.16710780996922509, 0.14495260038857935, 0.1523376702487946, 0.10626019976109023, 0.11160232273870782, 0.0988363420711699, 0.13866226536200174, 0.12425039519844053, 0.12936174240257164, 0.1441259842252803, 0.15250992344258824, 0.1523376702487946, 0.14495260038857935, 0.11160232273870782, 0.10626019976109023, 0.13866226536200174, 0.0988363420711699, 0.12936174240257164, 0.12425039519844053, 0.15250992344258824, 0.1441259842252803, 0.1671078099692252, 0.10626019976109033, 0.11160232273870793, 0.12425039519844056, 0.1293617424025717, 0.09883634207116994, 0.13866226536200194, 0.14412598422528042, ...]"
1,Z2,16,"[T_9, T_65, T_306, T_526, T_547, T_559, T_574, T_586, T_601, T_616, T_628, T_645, T_651, T_668, T_674, T_696]","[2.8471053474780827, -0.13293527926023815, -0.15753375159129723, -0.17811696595829796, -0.13999632264745243, -0.15281805234930942, -0.13999632264745246, -0.15281805234930945, -0.0835254057322433, -0.110941292194991, -0.11421199645458707, -0.12832548556368464, -0.11421199645458717, -0.12832548556368473, -0.1294816515181696, -0.14407388478161626]"
2,Z0,16,"[T_1, T_37, T_65, T_78, T_102, T_120, T_138, T_156, T_174, T_195, T_209, T_234, T_241, T_266, T_273, T_302]","[3.3866159975855386, -0.19597087956222953, -0.13293527926023815, -0.15753375159129723, -0.139211099222233, -0.17035513483381104, -0.13921109922223307, -0.1703551348338111, -0.1071600158191113, -0.1155422597736433, -0.13025744866044014, -0.13995040339462847, -0.1302574486604402, -0.13995040339462855, -0.14805676163167628, -0.17359014345771293]"
3,Z15,16,"[T_36, T_302, T_518, T_696, T_829, T_950, T_1035, T_1114, T_1168, T_1195, T_1211, T_1220, T_1224, T_1229, T_1231, T_1232]","[2.367041902369186, -0.17359014345771293, -0.14805676163167628, -0.1440738847816

In [7]:
# Compute Pauli-duplication ratio for JW and BK

from collections import defaultdict
import pandas as pd

from openfermion.ops import FermionOperator
from openfermion.transforms import jordan_wigner, bravyi_kitaev, normal_ordered


def pauli_support(qubit_op, tol=1e-12, include_identity=True):
    """
    Return the set of Pauli strings with nonzero coefficients.
    """
    qubit_op.compress(abs_tol=tol)

    support = set()

    for pauli_key, coeff in qubit_op.terms.items():
        if abs(coeff) <= tol:
            continue

        if not include_identity and pauli_key == ():
            continue

        support.add(pauli_key)

    return support


def map_fermion_to_qubit(op, mapping="JW", n_qubits=None):
    """
    Map a FermionOperator to a QubitOperator using JW or BK.
    """
    mapping = mapping.upper()

    if mapping == "JW":
        qop = jordan_wigner(op)

    elif mapping == "BK":
        if n_qubits is None:
            raise ValueError("n_qubits is required for BK.")

        try:
            qop = bravyi_kitaev(op, n_qubits=n_qubits)
        except TypeError:
            qop = bravyi_kitaev(op, n_qubits)

    else:
        raise ValueError("mapping must be 'JW' or 'BK'.")

    qop.compress(abs_tol=1e-12)
    return qop


def pauli_duplication_ratio(
    fermionic_terms,
    mapping="JW",
    n_qubits=None,
    include_identity=True,
    tol=1e-12,
):
    """
    Compute

        sum_alpha #mapping(H_alpha) / #mapping(H)

    where H = sum_alpha H_alpha.

    Also returns a duplicate-use table.
    """

    H_full = FermionOperator.zero()

    numerator = 0
    union_support = set()
    pauli_usage = defaultdict(list)

    for alpha, H_alpha in enumerate(fermionic_terms):
        H_full += H_alpha

        Q_alpha = map_fermion_to_qubit(
            H_alpha,
            mapping=mapping,
            n_qubits=n_qubits,
        )

        support_alpha = pauli_support(
            Q_alpha,
            tol=tol,
            include_identity=include_identity,
        )

        numerator += len(support_alpha)
        union_support |= support_alpha

        for pauli_key, coeff in Q_alpha.terms.items():
            if abs(coeff) <= tol:
                continue

            if not include_identity and pauli_key == ():
                continue

            pauli_usage[pauli_key].append(
                {
                    "vertex": f"T_{alpha}",
                    "coefficient": coeff,
                }
            )

    H_full = normal_ordered(H_full)
    H_full.compress(abs_tol=tol)

    Q_full = map_fermion_to_qubit(
        H_full,
        mapping=mapping,
        n_qubits=n_qubits,
    )

    full_support = pauli_support(
        Q_full,
        tol=tol,
        include_identity=include_identity,
    )

    denominator = len(full_support)

    duplication_ratio = numerator / denominator
    union_ratio = numerator / len(union_support)

    summary_df = pd.DataFrame(
        [
            {
                "mapping": mapping.upper(),
                "include_identity": include_identity,
                "sum_alpha_number_of_Pauli_strings": numerator,
                "number_of_unique_Pauli_strings_before_cancellation": len(union_support),
                "number_of_Pauli_strings_in_full_H": denominator,
                "duplication_ratio": duplication_ratio,
                "raw_reuse_ratio_before_cancellation": union_ratio,
            }
        ]
    )

    duplicate_rows = []

    for pauli_key, appearances in pauli_usage.items():
        if len(appearances) <= 1:
            continue

        combined_coeff = Q_full.terms.get(pauli_key, 0.0)

        duplicate_rows.append(
            {
                "Pauli_string": format_qubit_term(pauli_key),
                "number_of_appearances": len(appearances),
                "appears_in_vertices": [x["vertex"] for x in appearances],
                "individual_coefficients": [complex(x["coefficient"]) for x in appearances],
                "combined_coefficient_in_full_H": complex(combined_coeff),
                "survives_in_full_H": abs(combined_coeff) > tol,
            }
        )

    duplicate_df = pd.DataFrame(duplicate_rows)

    if len(duplicate_df) > 0:
        duplicate_df = duplicate_df.sort_values(
            "number_of_appearances",
            ascending=False,
        ).reset_index(drop=True)

    return summary_df, duplicate_df


# ------------------------------------------------------------
# Run for JW
# ------------------------------------------------------------

n_qubits = molecule.n_qubits

jw_summary_df, jw_duplicate_df = pauli_duplication_ratio(
    hermitian_terms,
    mapping="JW",
    n_qubits=n_qubits,
    include_identity=True,
)

print("=== JW Pauli-duplication ratio ===")
display(jw_summary_df)

print("\n=== Duplicated JW Pauli strings ===")
display(jw_duplicate_df)


# ------------------------------------------------------------
# Optional: run for BK too
# ------------------------------------------------------------

bk_summary_df, bk_duplicate_df = pauli_duplication_ratio(
    hermitian_terms,
    mapping="BK",
    n_qubits=n_qubits,
    include_identity=True,
)

print("=== BK Pauli-duplication ratio ===")
display(bk_summary_df)

print("\n=== Duplicated BK Pauli strings ===")
display(bk_duplicate_df)

=== JW Pauli-duplication ratio ===


,mapping,include_identity,sum_alpha_number_of_Pauli_strings,number_of_unique_Pauli_strings_before_cancellation,number_of_Pauli_strings_in_full_H,duplication_ratio,raw_reuse_ratio_before_cancellation
0,JW,True,7977,3689,2329,3.425075,2.162375



=== Duplicated JW Pauli strings ===


,Pauli_string,number_of_appearances,appears_in_vertices,individual_coefficients,combined_coefficient_in_full_H,survives_in_full_H
0,I,137,"[T_0, T_1, T_5, T_9, T_12, T_15, T_18, T_21, T_24, T_27, T_29, T_31, T_32, T_33, T_34, T_35, T_36, T_37, T_65, T_78, T_102, T_120, T_138, T_156, T_174, T_195, T_209, T_234, T_241, T_266, T_273, T_302, T_306, T_330, T_343, T_361, T_376, T_394, T_415, T_429, T_450, T_457, T_479, T_486, T_514, T_518, T_526, T_547, T_559, T_574, T_586, T_601, T_616, T_628, T_645, T_651, T_668, T_674, T_696, T_699, T_714, T_723, T_738, T_753, T_765, T_779, T_785, T_799, T_805, T_826, T_829, T_835, T_853, T_862, T_873, T_881, T_891, T_905, T_915, T_931, T_937, T_950, T_953, T_964, T_970, T_980, T_991, T_1001, T_1015, T_1021, T_1032, T_1035, T_1040, T_1053, T_1061, T_1071, T_1079, T_1087, T_1096, T_1101, ...]","[(-79.18038708155783+0j), (-3.3866159975855386+0j), (-3.3866159975855386+0j), (-2.8471053474780827+0j), (-2.8471053474780827+0j), (-2.844898468653311+0j), (-2.844898468653311+0j), (-2.8448984686533123+0j), (-2.8448984686533123+0j), (-2.2442892707457056+0j), (-2.2442892707457056+0j), (-2.3583722598550354+0j), (-2.3583722598550354+0j), (-2.3583722598550363+0j), (-2.3583722598550363+0j), (-2.367041902369186+0j), (-2.367041902369186+0j), (0.19597087956222953+0j), (0.13293527926023815+0j), (0.15753375159129723+0j), (0.139211099222233+0j), (0.17035513483381104+0j), (0.13921109922223307+0j), (0.1703551348338111+0j), (0.1071600158191113+0j), (0.1155422597736433+0j), (0.13025744866044014+0j), (0.13995040339462847+0j), (0.1302574486604402+0j), (0.13995040339462855+0j), (0.14805676163167628+0j), (0.17359014345771293+0j), (0.15753375159129723+0j), (0.13293527926023815+0j), (0.17035513483381104+0j), (0.139211099222233+0j), (0.1703551348338111+0j), (0.13921109922223307+0j), (0.1155422597736433+0j), (0.1071600158191113+0j), (0.13995040339462847+0j), (0.13025744866044014+0j), (0.13995040339462855+0j), (0.1302574486604402+0j), (0.17359014345771293+0j), (0.14805676163167628+0j), (0.17811696595829796+0j), (0.13999632264745243+0j), (0.15281805234930942+0j), (0.13999632264745246+0j), (0.15281805234930945+0j), (0.0835254057322433+0j), (0.110941292194991+0j), (0.11421199645458707+0j), (0.12832548556368464+0j), (0.11421199645458717+0j), (0.12832548556368473+0j), (0.1294816515181696+0j), (0.14407388478161626+0j), (0.15281805234930942+0j), (0.13999632264745243+0j), (0.15281805234930945+0j), (0.13999632264745246+0j), (0.110941292194991+0j), (0.0835254057322433+0j), (0.12832548556368464+0j), (0.11421199645458707+0j), (0.12832548556368473+0j), (0.11421199645458717+0j), (0.14407388478161626+0j), (0.1294816515181696+0j), (0.16710780996922509+0j), (0.14495260038857935+0j), (0.1523376702487946+0j), (0.10626019976109023+0j), (0.11160232273870782+0j), (0.0988363420711699+0j), (0.13866226536200174+0j), (0.12425039519844053+0j), (0.12936174240257164+0j), (0.1441259842252803+0j), (0.15250992344258824+0j), (0.1523376702487946+0j), (0.14495260038857935+0j), (0.11160232273870782+0j), (0.10626019976109023+0j), (0.13866226536200174+0j), (0.0988363420711699+0j), (0.12936174240257164+0j), (0.12425039519844053+0j), (0.15250992344258824+0j), (0.1441259842252803+0j), (0.1671078099692252+0j), (0.10626019976109033+0j), (0.11160232273870793+0j), (0.12425039519844056+0j), (0.1293617424025717+0j), (0.09883634207116994+0j), (0.13866226536200194+0j), (0.14412598422528042+0j), ...]",-105.472957+ 0.000000j,True
1,Z2,16,"[T_9, T_65, T_306, T_526, T_547, T_559, T_574, T_586, T_601, T_616, T_628, T_645, T_651, T_668, T_674, T_696]","[(2.8471053474780827+0j), (-0.13293527926023815+0j), (-0.15753375159129723+0j), (-0.17811696595829796+0j), (-0.13999632264745243+0j), (-0.15281805234930942+0j), (-0.13999632264745246+0j), (-0.15281805234930945+0j), (-0.0835254057322433+0j), (-0.110941292194991+0j), (-0.11421199645458707+0j), (-0.12832548556368464+0j), (-0.11421199645458717+0j), (-0.12832548556368473+0j), (-0.1294816515181696+0j), (-0.14407388478161626+0j)]",0.839793+ 0.0

=== BK Pauli-duplication ratio ===


,mapping,include_identity,sum_alpha_number_of_Pauli_strings,number_of_unique_Pauli_strings_before_cancellation,number_of_Pauli_strings_in_full_H,duplication_ratio,raw_reuse_ratio_before_cancellation
0,BK,True,7977,3689,2329,3.425075,2.162375



=== Duplicated BK Pauli strings ===


,Pauli_string,number_of_appearances,appears_in_vertices,individual_coefficients,combined_coefficient_in_full_H,survives_in_full_H
0,I,137,"[T_0, T_1, T_5, T_9, T_12, T_15, T_18, T_21, T_24, T_27, T_29, T_31, T_32, T_33, T_34, T_35, T_36, T_37, T_65, T_78, T_102, T_120, T_138, T_156, T_174, T_195, T_209, T_234, T_241, T_266, T_273, T_302, T_306, T_330, T_343, T_361, T_376, T_394, T_415, T_429, T_450, T_457, T_479, T_486, T_514, T_518, T_526, T_547, T_559, T_574, T_586, T_601, T_616, T_628, T_645, T_651, T_668, T_674, T_696, T_699, T_714, T_723, T_738, T_753, T_765, T_779, T_785, T_799, T_805, T_826, T_829, T_835, T_853, T_862, T_873, T_881, T_891, T_905, T_915, T_931, T_937, T_950, T_953, T_964, T_970, T_980, T_991, T_1001, T_1015, T_1021, T_1032, T_1035, T_1040, T_1053, T_1061, T_1071, T_1079, T_1087, T_1096, T_1101, ...]","[(-79.18038708155783+0j), (-3.3866159975855386+0j), (-3.3866159975855386+0j), (-2.8471053474780827+0j), (-2.8471053474780827+0j), (-2.844898468653311+0j), (-2.844898468653311+0j), (-2.8448984686533123+0j), (-2.8448984686533123+0j), (-2.2442892707457056+0j), (-2.2442892707457056+0j), (-2.3583722598550354+0j), (-2.3583722598550354+0j), (-2.3583722598550363+0j), (-2.3583722598550363+0j), (-2.367041902369186+0j), (-2.367041902369186+0j), (0.19597087956222953+0j), (0.13293527926023815+0j), (0.15753375159129723+0j), (0.139211099222233+0j), (0.17035513483381104+0j), (0.13921109922223307+0j), (0.1703551348338111+0j), (0.1071600158191113+0j), (0.1155422597736433+0j), (0.13025744866044014+0j), (0.13995040339462847+0j), (0.1302574486604402+0j), (0.13995040339462855+0j), (0.14805676163167628+0j), (0.17359014345771293+0j), (0.15753375159129723+0j), (0.13293527926023815+0j), (0.17035513483381104+0j), (0.139211099222233+0j), (0.1703551348338111+0j), (0.13921109922223307+0j), (0.1155422597736433+0j), (0.1071600158191113+0j), (0.13995040339462847+0j), (0.13025744866044014+0j), (0.13995040339462855+0j), (0.1302574486604402+0j), (0.17359014345771293+0j), (0.14805676163167628+0j), (0.17811696595829796+0j), (0.13999632264745243+0j), (0.15281805234930942+0j), (0.13999632264745246+0j), (0.15281805234930945+0j), (0.0835254057322433+0j), (0.110941292194991+0j), (0.11421199645458707+0j), (0.12832548556368464+0j), (0.11421199645458717+0j), (0.12832548556368473+0j), (0.1294816515181696+0j), (0.14407388478161626+0j), (0.15281805234930942+0j), (0.13999632264745243+0j), (0.15281805234930945+0j), (0.13999632264745246+0j), (0.110941292194991+0j), (0.0835254057322433+0j), (0.12832548556368464+0j), (0.11421199645458707+0j), (0.12832548556368473+0j), (0.11421199645458717+0j), (0.14407388478161626+0j), (0.1294816515181696+0j), (0.16710780996922509+0j), (0.14495260038857935+0j), (0.1523376702487946+0j), (0.10626019976109023+0j), (0.11160232273870782+0j), (0.0988363420711699+0j), (0.13866226536200174+0j), (0.12425039519844053+0j), (0.12936174240257164+0j), (0.1441259842252803+0j), (0.15250992344258824+0j), (0.1523376702487946+0j), (0.14495260038857935+0j), (0.11160232273870782+0j), (0.10626019976109023+0j), (0.13866226536200174+0j), (0.0988363420711699+0j), (0.12936174240257164+0j), (0.12425039519844053+0j), (0.15250992344258824+0j), (0.1441259842252803+0j), (0.1671078099692252+0j), (0.10626019976109033+0j), (0.11160232273870793+0j), (0.12425039519844056+0j), (0.1293617424025717+0j), (0.09883634207116994+0j), (0.13866226536200194+0j), (0.14412598422528042+0j), ...]",-105.472957+ 0.000000j,True
1,Z2,16,"[T_9, T_65, T_306, T_526, T_547, T_559, T_574, T_586, T_601, T_616, T_628, T_645, T_651, T_668, T_674, T_696]","[(2.8471053474780827+0j), (-0.13293527926023815+0j), (-0.15753375159129723+0j), (-0.17811696595829796+0j), (-0.13999632264745243+0j), (-0.15281805234930942+0j), (-0.13999632264745246+0j), (-0.15281805234930945+0j), (-0.0835254057322433+0j), (-0.110941292194991+0j), (-0.11421199645458707+0j), (-0.12832548556368464+0j), (-0.11421199645458717+0j), (-0.12832548556368473+0j), (-0.1294816515181696+0j), (-0.14407388478161626+0j)]",0.839793+ 0.0